# Data Analysis



### Importing Required Libraries

In [1]:
import pandas as pd
import numpy as np

### Loading Cleaned Dataset

In [2]:
# Load cleaned dataset
df = pd.read_csv("../data/silver/online_retail_cleaned.csv", parse_dates=["InvoiceDate"])
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Cancelled,ProductCategory,TotalPrice,Year,Month,Day,Hour,Weekend,ExpenseCategory
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,Decor,15.30,2010,12,1,8,False,Medium
1,536365,71053,WHITE METAL LANTERN,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,Lighting,20.34,2010,12,1,8,False,Medium
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,Decor,22.00,2010,12,1,8,False,Medium
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,Kitchen,20.34,2010,12,1,8,False,Medium
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,Decor,20.34,2010,12,1,8,False,Medium


### Basic Dataset Overview

In [3]:
# Shape
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Data types
df.info()

# Summary statistics
df.describe()

# Unique counts
print("Unique Customers:", df['CustomerID'].nunique())
print("Unique Products:", df['StockCode'].nunique())
print("Countries:", df['Country'].nunique())
print(df['Country'].value_counts().head())

Rows: 535278
Columns: 17
<class 'pandas.DataFrame'>
RangeIndex: 535278 entries, 0 to 535277
Data columns (total 17 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   InvoiceNo        535278 non-null  str           
 1   StockCode        535278 non-null  str           
 2   Description      534708 non-null  str           
 3   Quantity         535278 non-null  float64       
 4   InvoiceDate      535278 non-null  datetime64[us]
 5   UnitPrice        535278 non-null  float64       
 6   CustomerID       401604 non-null  float64       
 7   Country          535278 non-null  str           
 8   Cancelled        535278 non-null  bool          
 9   ProductCategory  535278 non-null  str           
 10  TotalPrice       535278 non-null  float64       
 11  Year             535278 non-null  int64         
 12  Month            535278 non-null  int64         
 13  Day              535278 non-null  int64         
 14  Hour  

### Extra Cleaning Steps

1. Replacing Country Abbreviations: 

In [4]:
# Replace EIRE and RSA with full country names
df["Country"] = df["Country"].replace({
    "EIRE": "Ireland",
    "RSA": "South Africa"
})

df["Country"].value_counts().head(10)

Country
United Kingdom    488937
Germany             9480
France              8541
Ireland             8184
Spain               2528
Netherlands         2371
Belgium             2069
Switzerland         1994
Portugal            1510
Australia           1258
Name: count, dtype: int64

2. Deleting Records with Invalid Country Names:

In [5]:
invalid_countries = ["European Community", "Unspecified", "Channel Islands"]

df = df[~df["Country"].isin(invalid_countries)].copy()

df["Country"].value_counts().head(10)

Country
United Kingdom    488937
Germany             9480
France              8541
Ireland             8184
Spain               2528
Netherlands         2371
Belgium             2069
Switzerland         1994
Portugal            1510
Australia           1258
Name: count, dtype: int64

3. Deleting Records with Product Category - Other

In [6]:
df = df[df["ProductCategory"] != "Other"].copy()
df["ProductCategory"].value_counts()

ProductCategory
Decor          135852
Kitchen         57874
Accessories     35351
Home            12604
Lighting         1511
Office            945
Name: count, dtype: int64

### Delete All Records Where CustomerID = -1

In [7]:
df = df[df["CustomerID"] != -1].copy()

In [8]:
df.to_csv('../data/gold/latest_clean_dataset.csv', index=False)


# Core Analysis

### Monthly Revenue

In [9]:
monthly_revenue = (
    df.groupby(["Year", "Month"])["TotalPrice"]
      .sum()
      .reset_index()
)

monthly_revenue.to_csv("../data/gold/monthly_revenue.csv", index=False)

monthly_revenue.head()

,Year,Month,TotalPrice
0,2010,12,409320.66
1,2011,1,325940.52
2,2011,2,245411.44
3,2011,3,330381.88
4,2011,4,246361.21


In [10]:
monthly_revenue.to_csv("../data/gold/monthly_revenue.csv", index=False)


### Country Revenue

In [11]:
country_revenue = (
    df.groupby("Country")["TotalPrice"]
      .sum()
      .reset_index()
)

country_revenue.head()


,Country,TotalPrice
0,Australia,67862.46
1,Austria,3695.22
2,Bahrain,227.10
3,Belgium,19294.15
4,Brazil,763.86


In [12]:

country_revenue.to_csv("../data/gold/country_revenue.csv", index=False)

### Product Revenue

In [13]:
product_revenue = (
    df.groupby(["StockCode", "Description", "ProductCategory"])
      .agg(
          TotalQuantity=("Quantity", "sum"),
          TotalRevenue=("TotalPrice", "sum")
      )
      .reset_index()
)

product_revenue.head()

,StockCode,Description,ProductCategory,TotalQuantity,TotalRevenue
0,10123C,HEARTS WRAPPING TAPE,Decor,5.0,3.25
1,15060B,FAIRY CAKE DESIGN UMBRELLA,Kitchen,492.0,1811.30
2,15060b,FAIRY CAKE DESIGN UMBRELLA,Kitchen,3.0,24.87
3,16020C,CLEAR STATIONERY BOX SET,Decor,157.0,69.90
4,16046,TEATIME PEN CASE & PENS,Kitchen,65.0,55.25


In [14]:
product_revenue.to_csv("../data/gold/product_revenue.csv", index=False)

### Category Revenue

In [15]:
category_revenue = (
    df.groupby("ProductCategory")["TotalPrice"]
      .sum()
      .reset_index()
)

category_revenue.head()

,ProductCategory,TotalPrice
0,Accessories,815370.87
1,Decor,2517685.79
2,Home,371201.08
3,Kitchen,1353717.22
4,Lighting,31915.53


In [16]:
category_revenue.to_csv("../data/gold/category_revenue.csv", index=False)

### Hourly Sales

In [17]:
hourly_sales = (
    df.groupby("Hour")["TotalPrice"]
      .sum()
      .reset_index()
)

hourly_sales.head(25)

,Hour,TotalPrice
0,6,120.20
1,7,15924.54
2,8,152399.57
3,9,402832.50
4,10,697601.57
5,11,611455.82
6,12,690285.93
7,13,633403.88
8,14,574020.65
9,15,639455.09


In [18]:
hourly_sales.to_csv("../data/gold/hourly_sales.csv", index=False)

### Weekend Sales

In [19]:
weekend_sales = (
    df.groupby("Weekend")["TotalPrice"]
      .sum()
      .reset_index()
)


weekend_sales.head()

,Weekend,TotalPrice
0,False,4707064.37
1,True,398911.25


In [20]:
weekend_sales.to_csv("../data/gold/weekend_sales.csv", index=False)

# Customer Analysis

### Customer Value Table

In [21]:
customer_value = (
    df.groupby("CustomerID")
      .agg(
          TotalRevenue=("TotalPrice", "sum"),
          OrderCount=("InvoiceNo", "nunique"),
          TotalQuantity=("Quantity", "sum")
      )
      .reset_index()
)

customer_value["AOV"] = customer_value["TotalRevenue"] / customer_value["OrderCount"]
customer_value["RevenueRank"] = customer_value["TotalRevenue"].rank(pct=True)

def segment_customer(p):
    if p >= 0.80:
        return "High Value"
    elif p >= 0.20:
        return "Mid Value"
    else:
        return "Low Value"

customer_value["Segment"] = customer_value["RevenueRank"].apply(segment_customer)

customer_value.head()


,CustomerID,TotalRevenue,OrderCount,TotalQuantity,AOV,RevenueRank,Segment
0,12347.0,1764.08,7,1220.0,252.011429,0.900903,High Value
1,12348.0,586.76,3,1040.0,195.586667,0.673954,Mid Value
2,12349.0,801.40,1,334.0,801.400000,0.744297,Mid Value
3,12350.0,20.40,1,24.0,20.400000,0.019962,Low Value
4,12352.0,662.89,8,279.0,82.861250,0.701521,Mid Value


In [22]:
customer_value.to_csv("../data/gold/customer_value.csv", index=False)

### Customer Orders Table

In [23]:
customer_orders = (
    df.groupby(["CustomerID", "InvoiceNo"])
      .agg(
          OrderRevenue=("TotalPrice", "sum"),
          OrderQuantity=("Quantity", "sum"),
          OrderDate=("InvoiceDate", "min")
      )
      .reset_index()
)

customer_orders.head()

,CustomerID,InvoiceNo,OrderRevenue,OrderQuantity,OrderDate
0,12347.0,537626,185.05,147.0,2010-12-07 14:57:00
1,12347.0,542237,221.49,167.0,2011-01-26 14:30:00
2,12347.0,549222,215.35,189.0,2011-04-07 10:43:00
3,12347.0,556201,180.70,82.0,2011-06-09 13:01:00
4,12347.0,562032,253.95,165.0,2011-08-02 08:48:00


In [24]:
customer_orders.to_csv("../data/gold/customer_orders.csv", index=False)

### Customer Country Table 

In [25]:
customer_country = (
    df.groupby(["CustomerID", "Country"])
      .agg(
          TotalRevenue=("TotalPrice", "sum"),
          OrderCount=("InvoiceNo", "nunique")
      )
      .reset_index()
)

customer_country.head()

,CustomerID,Country,TotalRevenue,OrderCount
0,12347.0,Iceland,1764.08,7
1,12348.0,Finland,586.76,3
2,12349.0,Italy,801.40,1
3,12350.0,Norway,20.40,1
4,12352.0,Norway,662.89,8


In [26]:
customer_country.to_csv("../data/gold/customer_country.csv", index=False)

### Customer Activity Table

In [27]:
customer_activity = (
    df.groupby(["CustomerID", "Hour", "Day", "Weekend"])
      .agg(
          Orders=("InvoiceNo", "nunique"),
          Revenue=("TotalPrice", "sum")
      )
      .reset_index()
)

customer_activity.head(15)

,CustomerID,Hour,Day,Weekend,Orders,Revenue
0,12347.0,8,2,False,1,253.95
1,12347.0,10,7,False,1,215.35
2,12347.0,12,31,False,1,685.76
3,12347.0,13,9,False,1,180.70
4,12347.0,14,7,False,1,185.05
5,12347.0,14,26,False,1,221.49
6,12347.0,15,7,False,1,21.78
7,12348.0,10,5,False,1,17.00
8,12348.0,10,25,False,1,62.16
9,12348.0,19,16,False,1,507.60


In [28]:
customer_activity.to_csv("../data/gold/customer_activity.csv", index=False)

# Time Series Analysis

### Daily Revenue

In [29]:
daily_revenue = (
    df.groupby(df["InvoiceDate"].dt.date)["TotalPrice"]
      .sum()
      .reset_index(name="TotalRevenue")
)

daily_revenue.head()


,InvoiceDate,TotalRevenue
0,2010-12-01,26683.03
1,2010-12-02,22125.22
2,2010-12-03,27125.48
3,2010-12-05,15070.18
4,2010-12-06,24471.92


In [30]:
daily_revenue.to_csv("../data/gold/daily_revenue.csv", index=False)

###  Weekly Revenue

In [31]:
df["Week"] = df["InvoiceDate"].dt.isocalendar().week

weekly_revenue = (
    df.groupby(["Year", "Week"])["TotalPrice"]
      .sum()
      .reset_index()
)

weekly_revenue.head()


,Year,Week,TotalPrice
0,2010,48,91003.91
1,2010,49,159505.63
2,2010,50,108847.54
3,2010,51,49963.58
4,2011,1,69250.53


In [32]:
weekly_revenue.to_csv("../data/gold/weekly_revenue.csv", index=False)

### Seasonality Table

In [33]:
seasonality = (
    df.groupby(["Month", "Day", "Hour"])["TotalPrice"]
      .sum()
      .reset_index()
)

seasonality.head(20)


,Month,Day,Hour,TotalPrice
0,1,4,10,938.68
1,1,4,11,829.67
2,1,4,12,881.30
3,1,4,13,2344.00
4,1,4,14,1400.44
5,1,4,15,804.35
6,1,4,16,321.25
7,1,5,9,283.50
8,1,5,10,2519.71
9,1,5,11,5319.73


In [34]:
seasonality.to_csv("../data/gold/seasonality.csv", index=False)

# RFM Segmentation

### RFM Segments & RFM Scores

In [35]:
# --- RFM BASE TABLE ---

# Snapshot date = day after last transaction
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm_scores = (
    df.groupby("CustomerID")
      .agg(
          Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
          Frequency=("InvoiceNo", "nunique"),
          Monetary=("TotalPrice", "sum")
      )
      .reset_index()
)

# --- R, F, M RANKS (FIXED VERSION) ---

# Recency: lower days = better → labels reversed
rfm_scores["R_rank"] = pd.qcut(
    rfm_scores["Recency"],
    4,
    labels=[4, 3, 2, 1]
)

# Frequency: use rank to avoid duplicate bin edges
rfm_scores["F_rank"] = pd.qcut(
    rfm_scores["Frequency"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
)

# Monetary: same trick as Frequency
rfm_scores["M_rank"] = pd.qcut(
    rfm_scores["Monetary"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]



    
)

# --- RFM SCORE ---

rfm_scores["RFM_Score"] = (
    rfm_scores["R_rank"].astype(int)
    + rfm_scores["F_rank"].astype(int)
    + rfm_scores["M_rank"].astype(int)
)

# --- OPTIONAL: SEGMENT LABELS ---

def rfm_segment(score):
    if score >= 10:
        return "Champion"
    elif score >= 8:
        return "Loyal"
    elif score >= 6:
        return "Potential Loyalist"
    elif score >= 4:
        return "At Risk"
    else:
        return "Hibernating"

rfm_scores["Segment"] = rfm_scores["RFM_Score"].apply(rfm_segment)
rfm_scores.head()


,CustomerID,Recency,Frequency,Monetary,R_rank,F_rank,M_rank,RFM_Score,Segment
0,12347.0,2,7,1764.08,4,4,4,12,Champion
1,12348.0,249,3,586.76,1,3,3,7,Potential Loyalist
2,12349.0,19,1,801.40,3,1,3,7,Potential Loyalist
3,12350.0,310,1,20.40,1,1,1,3,Hibernating
4,12352.0,36,8,662.89,3,4,3,10,Champion


In [36]:
# --- SAVE TO GOLD ---

# Full RFM scores table
rfm_scores.to_csv("../data/gold/rfm_scores.csv", index=False)